# HierarchicalDet — Kaggle Training Notebook

Trains all three tiers of HierarchicalDet on the DENTEX dataset.

**Session strategy** (Kaggle T4 = 12h limit, ~12–20h per tier):
- Each session trains one tier
- Checkpoints are saved to `/kaggle/working/output/` which becomes notebook output
- Next session: attach previous output as input dataset → resumes from checkpoint

**Workflow across sessions:**
```
Session 1: Train tier1 → save output/tier1/model_final.pth
Session 2: Attach Session 1 output → generate noisy boxes → train tier2
Session 3: Attach Session 2 output → generate noisy boxes → train tier3
Session 4: Evaluation + extensions
```

**Before running**: set `GITHUB_REPO` and `TIER_TO_TRAIN` in cell 2.

In [ ]:
# ── CONFIGURATION — edit these before running ─────────────────────────────────

GITHUB_REPO   = 'https://github.com/YOUR_USERNAME/HierarchicalDet'  # your fork
TIER_TO_TRAIN = 1   # 1, 2, or 3 — which tier to train this session
MAX_ITER      = 40000
NUM_GPUS      = 1

# If resuming: path to previous session's output directory mounted as Kaggle input
# e.g. '/kaggle/input/hierarchicaldet-tier1/output'
# Set to None if this is a fresh start (tier 1, no prior checkpoint)
PREV_OUTPUT_DIR = None  # e.g. '/kaggle/input/hierarchicaldet-tier1/output'

# If you uploaded DENTEX zips as a Kaggle dataset, set this to its mount path.
# e.g. '/kaggle/input/dentex-dataset'
# Set to None to download from HuggingFace instead (adds ~30 min per session).
KAGGLE_DENTEX_INPUT = None  # e.g. '/kaggle/input/dentex-dataset'

# Hugging Face token (only needed if KAGGLE_DENTEX_INPUT is None)
HF_TOKEN = None

import os
WORK_DIR    = '/kaggle/working'
REPO_DIR    = f'{WORK_DIR}/HierarchicalDet'
DATA_ROOT   = f'{WORK_DIR}/sorted/challenge'
OUTPUT_DIR  = f'{WORK_DIR}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Tier to train:       {TIER_TO_TRAIN}')
print(f'Previous output:     {PREV_OUTPUT_DIR}')
print(f'DENTEX input:        {KAGGLE_DENTEX_INPUT or "HuggingFace download"}')

In [ ]:
# ── 0. Hardware check ─────────────────────────────────────────────────────────
import subprocess, time, shutil
import torch

print('=== HARDWARE ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!free -h | grep Mem
!df -h /kaggle/working | tail -1
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

In [ ]:
# ── 1. Session timer — warns when < 90 min remain ────────────────────────────
import threading, time as _time

SESSION_START = _time.time()
SESSION_LIMIT_HOURS = 11.5   # stop training with 30 min buffer before Kaggle cuts off

def _timer_thread():
    while True:
        elapsed = (_time.time() - SESSION_START) / 3600
        remaining = SESSION_LIMIT_HOURS - elapsed
        if remaining <= 1.5:
            print(f'\n⚠️  WARNING: {remaining:.1f}h remaining — save checkpoints soon!')
        if remaining <= 0:
            print('\n🛑 Session limit reached — training will be interrupted.')
            break
        _time.sleep(1800)  # check every 30 min

t = threading.Thread(target=_timer_thread, daemon=True)
t.start()
print(f'Session timer started. Limit: {SESSION_LIMIT_HOURS}h')

In [ ]:
# ── 2. Clone reproduction repo ────────────────────────────────────────────────
import os

if os.path.exists(REPO_DIR):
    print(f'Repo already exists: {REPO_DIR}')
    !cd {REPO_DIR} && git pull
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

# Also clone the official HierarchicalDet repo for the actual model code
OFFICIAL_DIR = f'{WORK_DIR}/HierarchicalDet_official'
if not os.path.exists(OFFICIAL_DIR):
    !git clone https://github.com/ibrahimethemhamamci/HierarchicalDet {OFFICIAL_DIR}
    print('Official repo cloned.')
else:
    print('Official repo already exists.')

# Copy reproduction scripts into official repo
import shutil, glob
for f in glob.glob(f'{REPO_DIR}/*.py') + glob.glob(f'{REPO_DIR}/*.sh') + glob.glob(f'{REPO_DIR}/*.md'):
    shutil.copy(f, OFFICIAL_DIR)
print('Reproduction scripts copied to official repo.')
%cd {OFFICIAL_DIR}

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
# Kaggle has PyTorch pre-installed. We only need the additional deps.
# detectron2 is BUNDLED — build from source, do NOT pip install detectron2.

import time
t0 = time.time()

!pip install -q fvcore iopath omegaconf timm scipy termcolor yacs tabulate cloudpickle pycocotools huggingface_hub

# Build bundled detectron2
!pip install -q -e {OFFICIAL_DIR}

import sys
sys.path.insert(0, OFFICIAL_DIR)

import detectron2
print(f'detectron2: {detectron2.__version__}')
print(f'Install time: {time.time()-t0:.0f}s')

In [ ]:
# ── 4. Download Swin-B backbone ───────────────────────────────────────────────
import urllib.request, pickle, os

os.makedirs(f'{OFFICIAL_DIR}/models', exist_ok=True)
pkl_path = f'{OFFICIAL_DIR}/models/swin_base_patch4_window7_224_22k.pkl'

if not os.path.exists(pkl_path):
    url = ('https://github.com/SwinTransformer/storage/releases/download/'
           'v1.0.0/swin_base_patch4_window7_224_22k.pth')
    pth_path = pkl_path.replace('.pkl', '.pth')
    print('Downloading Swin-B backbone (~340 MB)...')
    t0 = time.time()
    urllib.request.urlretrieve(url, pth_path)
    print(f'Downloaded in {time.time()-t0:.0f}s')

    ckpt = torch.load(pth_path, map_location='cpu')
    d2_ckpt = {'model': ckpt.get('model', ckpt),
               '__author__': 'third_party', 'matching_heuristics': True}
    with open(pkl_path, 'wb') as f:
        pickle.dump(d2_ckpt, f)
    print(f'Backbone saved: {pkl_path}')
else:
    print(f'Backbone already present.')

In [ ]:
# ── 5. Set up DENTEX dataset ──────────────────────────────────────────────────
import os, zipfile, time

DENTEX_RAW = f'{WORK_DIR}/sorted/dentex_raw/DENTEX'
CHALLENGE  = f'{WORK_DIR}/sorted/challenge'

if os.path.exists(f'{CHALLENGE}/train_merged_disease_coco3class_onlyd_fixed.json'):
    print('Dataset already organized. Skipping.')

elif KAGGLE_DENTEX_INPUT:
    # ── Fast path: use pre-uploaded Kaggle dataset (no download needed) ──────
    print(f'Using mounted Kaggle dataset: {KAGGLE_DENTEX_INPUT}')
    os.makedirs(DENTEX_RAW, exist_ok=True)

    # Copy validation_triple.json (small, needed by organize_dataset.py)
    import shutil
    val_json_src = f'{KAGGLE_DENTEX_INPUT}/validation_triple.json'
    if os.path.exists(val_json_src):
        shutil.copy(val_json_src, f'{DENTEX_RAW}/validation_triple.json')

    # Extract zips from mounted input (read-only, must extract to working dir)
    for zname in ['training_data.zip', 'validation_data.zip', 'test_data.zip']:
        zpath = f'{KAGGLE_DENTEX_INPUT}/{zname}'
        if os.path.exists(zpath):
            print(f'Extracting {zname}...')
            t0 = time.time()
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(f'{WORK_DIR}/sorted')
            print(f'  Done in {time.time()-t0:.0f}s')
        else:
            print(f'  WARNING: {zname} not found in {KAGGLE_DENTEX_INPUT}')

    print('Extraction complete. Organizing...')
    !python {OFFICIAL_DIR}/organize_dataset.py

else:
    # ── Slow path: download from HuggingFace (~30 min) ───────────────────────
    from huggingface_hub import snapshot_download
    print('Downloading DENTEX dataset (~11.8 GB) from HuggingFace...')
    print('TIP: Upload zips as a Kaggle dataset to skip this next time.')
    t0 = time.time()
    if HF_TOKEN:
        from huggingface_hub import login
        login(HF_TOKEN)
    snapshot_download(
        repo_id='ibrahimhamamci/DENTEX',
        repo_type='dataset',
        local_dir=DENTEX_RAW,
        ignore_patterns=['*.git*', '.gitattributes'],
    )
    print(f'Download complete in {(time.time()-t0)/60:.1f} min')

    for zname in ['training_data.zip', 'validation_data.zip', 'test_data.zip']:
        zpath = f'{DENTEX_RAW}/{zname}'
        if os.path.exists(zpath):
            print(f'Extracting {zname}...')
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(f'{WORK_DIR}/sorted')
    print('Extraction complete. Organizing...')
    !python {OFFICIAL_DIR}/organize_dataset.py

print('\nDataset ready:')
!ls {CHALLENGE}

In [ ]:
# ── 6. Restore checkpoints from previous session ──────────────────────────────
import shutil, os

tier_output = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}'
os.makedirs(tier_output, exist_ok=True)

if PREV_OUTPUT_DIR:
    print(f'Restoring checkpoints from: {PREV_OUTPUT_DIR}')
    for item in os.listdir(PREV_OUTPUT_DIR):
        src = os.path.join(PREV_OUTPUT_DIR, item)
        dst = os.path.join(f'{OFFICIAL_DIR}/output', item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print(f'  Restored: {item}')
    print('Checkpoint restore complete.')
else:
    print('No previous output to restore (fresh start).')

# Show what we have
!find {OFFICIAL_DIR}/output -name '*.pth' 2>/dev/null | head -10 || echo 'No checkpoints yet.'

In [ ]:
# ── 7. Set environment variables ──────────────────────────────────────────────
import os

os.environ['DATA_ROOT']   = f'{WORK_DIR}/sorted/challenge'
os.environ['DENTEX_TIER'] = f'tier{TIER_TO_TRAIN}'
os.environ['USE_NOISY_BOXES'] = '0'   # overridden below for tiers 2+

TIER_MAP = {1: 'tier1', 2: 'tier2', 3: 'tier3'}
CONFIG   = f'{OFFICIAL_DIR}/configs/diffdet.custom.swinbase.nonpretrain.yaml'

# Determine weights init and noisy box paths
if TIER_TO_TRAIN == 1:
    WEIGHTS = f'{OFFICIAL_DIR}/models/swin_base_patch4_window7_224_22k.pkl'
    os.environ['USE_NOISY_BOXES'] = '0'
elif TIER_TO_TRAIN == 2:
    WEIGHTS = f'{OFFICIAL_DIR}/output/tier1/model_final.pth'
    NB_TRAIN = f'{OFFICIAL_DIR}/noisy_boxes/tier1_train_boxes.json'
    NB_VAL   = f'{OFFICIAL_DIR}/noisy_boxes/tier1_val_boxes.json'
    if os.path.exists(NB_TRAIN):
        os.environ['NOISY_BOX_TRAIN'] = NB_TRAIN
        os.environ['NOISY_BOX_VAL']   = NB_VAL
        os.environ['USE_NOISY_BOXES'] = '1'
    else:
        print('WARNING: noisy boxes not found — running generate step first')
elif TIER_TO_TRAIN == 3:
    WEIGHTS = f'{OFFICIAL_DIR}/output/tier2/model_final.pth'
    NB_TRAIN = f'{OFFICIAL_DIR}/noisy_boxes/tier2_train_boxes.json'
    NB_VAL   = f'{OFFICIAL_DIR}/noisy_boxes/tier2_val_boxes.json'
    if os.path.exists(NB_TRAIN):
        os.environ['NOISY_BOX_TRAIN'] = NB_TRAIN
        os.environ['NOISY_BOX_VAL']   = NB_VAL
        os.environ['USE_NOISY_BOXES'] = '1'
    else:
        print('WARNING: noisy boxes not found — running generate step first')

print(f'Tier:           {TIER_TO_TRAIN}')
print(f'Weights:        {WEIGHTS}')
print(f'Config:         {CONFIG}')
print(f'USE_NOISY_BOXES:{os.environ["USE_NOISY_BOXES"]}')
print(f'DATA_ROOT:      {os.environ["DATA_ROOT"]}')

In [ ]:
# ── 8. Generate noisy boxes (tiers 2 and 3 only) ─────────────────────────────
# Skip for tier 1 or if already generated

if TIER_TO_TRAIN > 1 and os.environ.get('USE_NOISY_BOXES') != '1':
    prev_tier = TIER_TO_TRAIN - 1
    prev_weights = f'{OFFICIAL_DIR}/output/tier{prev_tier}/model_final.pth'
    assert os.path.exists(prev_weights), f'Tier {prev_tier} checkpoint not found: {prev_weights}'

    os.makedirs(f'{OFFICIAL_DIR}/noisy_boxes', exist_ok=True)

    for split, split_json, split_imgs, out_file in [
        ('train',
         f'{WORK_DIR}/sorted/challenge/train_merged_disease_coco3class_onlyd_fixed.json',
         f'{WORK_DIR}/sorted/challenge/for_coco_disease_train',
         f'{OFFICIAL_DIR}/noisy_boxes/tier{prev_tier}_train_boxes.json'),
        ('val',
         f'{WORK_DIR}/sorted/challenge/test_merged_disease_coco3class.json',
         f'{WORK_DIR}/sorted/challenge/for_coco_disease_test',
         f'{OFFICIAL_DIR}/noisy_boxes/tier{prev_tier}_val_boxes.json'),
    ]:
        print(f'Generating noisy boxes for {split} split...')
        !python {OFFICIAL_DIR}/phase2_generate_noisy_boxes.py \
            --config-file {CONFIG} \
            --weights {prev_weights} \
            --tier {prev_tier - 1} \
            --ann-json {split_json} \
            --img-dir {split_imgs} \
            --out {out_file}

    os.environ['NOISY_BOX_TRAIN'] = f'{OFFICIAL_DIR}/noisy_boxes/tier{prev_tier}_train_boxes.json'
    os.environ['NOISY_BOX_VAL']   = f'{OFFICIAL_DIR}/noisy_boxes/tier{prev_tier}_val_boxes.json'
    os.environ['USE_NOISY_BOXES'] = '1'
    print('Noisy boxes generated.')
else:
    print(f'Tier {TIER_TO_TRAIN}: noisy box step skipped.')

In [ ]:
# ── 9. Train ──────────────────────────────────────────────────────────────────
import os

out_dir = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}'
os.makedirs(out_dir, exist_ok=True)

# Check if already completed
final_ckpt = f'{out_dir}/model_final.pth'
if os.path.exists(final_ckpt):
    print(f'model_final.pth already exists — tier {TIER_TO_TRAIN} already trained!')
    print('Set TIER_TO_TRAIN to the next tier, or proceed to evaluation.')
else:
    print(f'Starting tier {TIER_TO_TRAIN} training ({MAX_ITER} iterations)...')
    print(f'Estimated time: 12–20h on T4. Session limit: {SESSION_LIMIT_HOURS}h')
    print(f'Training will auto-resume if checkpoint exists in {out_dir}/')
    print('─' * 60)

    !python {OFFICIAL_DIR}/train_net_patched.py \
        --config-file {CONFIG} \
        --num-gpus {NUM_GPUS} \
        MODEL.WEIGHTS {WEIGHTS} \
        OUTPUT_DIR {out_dir} \
        SOLVER.MAX_ITER {MAX_ITER} \
        SEED 40244023

    if os.path.exists(final_ckpt):
        size_mb = os.path.getsize(final_ckpt) / 1e6
        print(f'\n✓ Training complete! Checkpoint: {final_ckpt} ({size_mb:.0f} MB)')
    else:
        print('\n⚠ Training stopped before completion (session limit or error).')
        print('Save the output directory and resume in a new session.')

In [ ]:
# ── 10. Evaluate ──────────────────────────────────────────────────────────────
# Run this after training completes to get mAP numbers.

final_ckpt = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}/model_final.pth'
if not os.path.exists(final_ckpt):
    print('No final checkpoint yet — run training cell first.')
else:
    eval_out = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}_eval'
    os.makedirs(eval_out, exist_ok=True)
    print(f'Evaluating tier {TIER_TO_TRAIN}...')

    !python {OFFICIAL_DIR}/train_net_patched.py \
        --config-file {CONFIG} \
        --eval-only \
        MODEL.WEIGHTS {final_ckpt} \
        OUTPUT_DIR {eval_out}

    # Parse and show results
    !python {OFFICIAL_DIR}/phase2_collect_results.py \
        --log-files {eval_out}/log.txt \
        --tier-names "Tier{TIER_TO_TRAIN}"

In [ ]:
# ── 11. Save outputs for next session ─────────────────────────────────────────
# Everything in /kaggle/working/ is saved as notebook output.
# After this cell: click "Save Version" → "Save & Run All" to commit outputs.
# Then in the next session, add this notebook's output as an input dataset.

import os, shutil

# Copy output/ and noisy_boxes/ to /kaggle/working/output so they're saved
src_output = f'{OFFICIAL_DIR}/output'
dst_output = f'{WORK_DIR}/output'

if os.path.exists(src_output):
    shutil.copytree(src_output, dst_output, dirs_exist_ok=True)
    print(f'Copied output/ to {dst_output}')

src_nb = f'{OFFICIAL_DIR}/noisy_boxes'
dst_nb = f'{WORK_DIR}/noisy_boxes'
if os.path.exists(src_nb):
    shutil.copytree(src_nb, dst_nb, dirs_exist_ok=True)
    print(f'Copied noisy_boxes/ to {dst_nb}')

# Summary
print('\n=== Files saved to /kaggle/working/ ===')
for root, dirs, files in os.walk(WORK_DIR + '/output'):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f'  {path.replace(WORK_DIR, "")} ({size_mb:.0f} MB)')

print('\n=== Next steps ===')
next_tier = TIER_TO_TRAIN + 1
if next_tier <= 3:
    print(f'1. Click "Save Version" → "Save & Run All" to commit these outputs')
    print(f'2. In a new session, add this notebook\'s output as input dataset')
    print(f'3. Set TIER_TO_TRAIN = {next_tier} and PREV_OUTPUT_DIR = "/kaggle/input/<dataset>/output"')
    print(f'4. Run all cells again')
else:
    print('All 3 tiers complete! Proceed to baselines and extensions.')
    print('Run: bash run_baselines.sh && bash run_extensions.sh')

In [ ]:
# ── 12. (Optional) Runtime benchmark ─────────────────────────────────────────
# Run after training to measure GPU inference speed.

final_ckpt = f'{OFFICIAL_DIR}/output/tier{TIER_TO_TRAIN}/model_final.pth'
if os.path.exists(final_ckpt):
    !python {OFFICIAL_DIR}/phase2_runtime_benchmark.py \
        --config-file {CONFIG} \
        --weights {final_ckpt} \
        --tier {TIER_TO_TRAIN - 1} \
        --n-images 20 \
        --device cuda \
        --out {WORK_DIR}/output/runtime_tier{TIER_TO_TRAIN}.json

    # Also CPU benchmark (small n)
    !python {OFFICIAL_DIR}/phase2_runtime_benchmark.py \
        --config-file {CONFIG} \
        --weights {final_ckpt} \
        --tier {TIER_TO_TRAIN - 1} \
        --n-images 5 \
        --device cpu \
        --out {WORK_DIR}/output/runtime_tier{TIER_TO_TRAIN}_cpu.json